# ESM2 full fine tuning


In [35]:
# Dependancies and libraries
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence
import torch.optim as optim

from transformers import AutoModel, AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification

import esm

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import math

from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold, train_test_split
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, matthews_corrcoef, roc_auc_score, confusion_matrix

from pathlib import Path

In [3]:
# ESM checkpoints
ESM = ['facebook/esm2_t48_15B_UR50D',
        'facebook/esm2_t36_3B_UR50D',
        'facebook/esm2_t33_650M_UR50D',
        'facebook/esm2_t30_150M_UR50D',
        'facebook/esm2_t12_35M_UR50D',
        'facebook/esm2_t6_8M_UR50D']

In [4]:
# Define checkpoint to be used
checkpoint = ESM[5]

In [14]:
# Create tokenizer objects
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

In [6]:
# Import target data
df_target= pd.read_csv("/Users/harry/Documents/Data Science MSc/PROJECT/MScProject/Target_1769.csv")

# Remove extraneous columns
df_target = df_target.iloc[:,0:5]

In [6]:
# Aggreate rows and update format
df_target = df_target.groupby(['Info_protein_id', 'Info_group']).agg(
    sequence=('Info_AA', ''.join), 
    label=('Class', list), 
    position=('Info_pos', list))

In [7]:
df_target.shape
df_target['label'].str.len().agg(['mean','max'])

mean     410.571429
max     1871.000000
Name: label, dtype: float64

In [8]:
# Create a list of sequences
sequences = df_target['sequence'].tolist()

# Instantiate the tokenizer using the AA sequence lists as input to the tokenizer to create tokenized sequences
inputs = tokenizer(
    sequences,
    padding=True,
    truncation=True,
    max_length=1024,
    return_tensors="pt"
)

# Put the model into evaluation mode
model.eval()

# Generate embeddings for the 
with torch.inference_mode():
    outputs = model(**inputs)

In [9]:
# Verify shape of embedding
outputs.last_hidden_state.size()

torch.Size([21, 1024, 320])

In [10]:
# Freeze the weights in the model
for param in model.parameters():
    param.requires_grad = False

In [7]:
# # Create a class column that checks whether the sequence contains a positive or negative epitope and apply a class column for the stratified grouped k fold
# df_lower["class"] = df_lower["label"].apply(lambda x: 1 if 1 in x else -1)

# # Check max length of sequences
# df_lower['label'].str.len().agg(['mean','max'])

Df contains sequences with length > ESM max input length (1024), sliding window will need to be applied once draft complete - same applies for target data

In [8]:
# # Info_group as the grouping variable and Class  / label as the stratification variable.
# X = df_lower.index
# y = df_lower['class']
# groups = df_lower['Info_group']

# # Instantiate GroupShuffleSplit instance to create grouped train/test splits, use 20% of the data for a hold out/test set
# gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)

# # Split the data into train/test splits, create indices to be used to assign train/test labels to the df
# train_cv_idx, test_idx = next(gss.split(X, y, groups))

# # Create a train/test column in the dataframe and set the values of the test rows to train or test
# df_lower.loc[test_idx, 'train_test'] = 'test'
# df_lower.loc[train_cv_idx, 'train_test'] = 'train'

# # Update X, y and groups with the remaining train_cv_idx indices to use in Statified Grouped k fold
# X_train, y_train, groups_train = X[train_cv_idx], y[train_cv_idx], groups[train_cv_idx]


In [9]:
# # Split into different folds ensuring stratification accross groups
# sgkf = StratifiedGroupKFold(n_splits=5)

# X_train_df = pd.DataFrame(index=range(0,len(df_lower)))
                                     
# for fold, (train_idx, val_idx) in enumerate(sgkf.split(X, y, groups)):
#     X_train_df.loc[train_idx, f'training_split {fold+1}'] = 1
#     X_train_df.loc[val_idx, f'training_split {fold+1}']= 2

# df_lower = df_lower.merge(X_train_df, left_index=True, right_index=True)

### Sliding window 

- Check whether a sequence is greater than 1024 residues in length
- If > 1024, create copy of sequence
- Create segments

In [8]:
def sliding_window(df, window_size=1024, stride=512):
    """
    Function to create 1024 length splits for proteins >1024 in length, using stride length of 512
    Inserts new splits into a datatable with split suffix
    """
    rows = []
    
    for _, row in df.iterrows():
        seq = row['sequence']
        seq_len = len(seq)
        
        if seq_len <= window_size:
            rows.append(row.copy())
            continue
            
        start = 0
        split = 1
        while start < seq_len:
            # End slicing variable
            end = start + window_size

            # Create copy of the row
            new_row = row.copy()
            # Slice amino acid sequence by start end index
            new_row['sequence'] = seq[start:end]
            
            # Slice columns containing lists
            for col in ['label', 'position']:
                new_row[col] = row[col][start:end]

            # Add split suffix to Info_protein_id
            new_row['Info_protein_id'] = new_row['Info_protein_id'] + '_' + str(split) 
            # Append window to row list 
            rows.append(new_row)
            
            # Stop after the final window
            if end >= seq_len:
                break
                
            start += stride
            split += 1
            
    return pd.DataFrame(rows).reset_index(drop=True)

### Fine Tuning

- Fine tune using higher level data

In [9]:
# Import lower level data
df_higher_stacked = pd.read_csv('/Users/harry/Documents/Data Science MSc/Higher_1783272.csv')

df_higher_stacked['Class'] = df_higher_stacked['Class'].fillna(-100).astype('int32')
df_higher_stacked['Class'] = df_higher_stacked['Class'].replace(-1, 0)

# Sort values before aggregation
df_higher_stacked = df_higher_stacked.sort_values(['Info_protein_id', 'Info_pos'])

# Aggregate columns for wide format
df_higher = df_higher_stacked.groupby(['Info_protein_id','Info_group'], as_index=False).agg(
    sequence=('Info_AA', ''.join), 
    label=('Class', list), 
    position=('Info_pos', list))

df_higher

,Info_protein_id,Info_group,sequence,label,position
0,1007216A,190.0,GADDVVDSSKSFVMENFSSYHGTKPGYVDSIQKGIQKPKSGTQGNY...,"[-100, -100, -100, -100, -100, -100, -100, -10...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14..."
1,1106184A,270.0,MKNYLSFGMFALLFALTFGTVNSVQAIAGPEWLLDRPSVNNSQLVV...,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, -100, ...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14..."
2,A26297,236.0,MAKNNTNRHYSLRKLKKGTASVAVALSVIGAGLVVNTNEVSARVFP...,"[-100, -100, -100, -100, -100, -100, -100, -10...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14..."
3,A32192,60.0,VKNNLRYGIRKHKLGAASVFLGTMIVVGMGQDKEAAASEQKTTTVE...,"[-100, -100, -100, -100, -100, -100, -100, -10...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14..."
4,A60328,393.0,MNQKIVVISSFYMLGAHSFSKAVYHNDRSVKLMKRIDINHQAQRFS...,"[-100, -100, -100, -100, -100, -100, -100, -10...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14..."
...,...,...,...,...,...
439,ZP_03596362.1,516.0,MERLQKVIAHAGVASRRKAEELIKEGKVKVNGKVVTELGVKVTGSD...,"[-100, -100, -100, -100, -100, -100, -100, -10...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14..."
440,ZP_03980798.1,453.0,MDIRFEQVDFTYQPNTPFEQRALFDINMTIKENSYTALVGHTGSGK...,"[-100, -100, -100, -100, -100, -100, -100, -10...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14..."
441,ZP_04069274.1,118.0,MNYMEDSSLDTLSIVNETDFPLYNNYTEPTIAPALIAVAPIAQYLA...,"[-100, -100, -100, -100, -100, -100, -100, -10...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14..."
442,ZP_05686172.1,434.0,MKKLVPLLLALLLLVAACGTGGKQSSDKSNGKLKVVTTNSILYDMA...,"[-100, -100, -100, -100, -100, -100, -100, -10...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14..."


In [11]:
# Preprocess long sequences 
df_higher = sliding_window(df_higher)

# Creating a train df with 80% of the rows
df_higher_train = df_higher.sample(frac=0.8, random_state=42).reset_index()

# Create validation df with remaining rows
df_higher_val = df_higher.drop(df_higher_train.index).reset_index()

In [12]:
def preprocess(data):
    """
    Preprocess each df row to tokenise the sequences and pad labels to match max length
    """
    inputs = tokenizer(
            data['sequence'],
            padding='max_length',
            truncation=True, 
            max_length=1026,
        )
    
    labels = data['label']

    # Remove first and last special tokens for tokens and attention mask
    input_ids = inputs['input_ids'][1:-1]
    attn_mask = inputs['attention_mask'][1:-1]

    # Pad labels to max length
    labels = labels + ([-100] * (1024 - len(labels)))

    return {'input_ids': input_ids,
            'attention_mask': attn_mask,
            'labels': labels}

preprocessed_train = df_higher_train.apply(preprocess, axis=1)
preprocessed_val = df_higher_val.apply(preprocess, axis=1)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)


In [30]:
# Define Trainer parameters
def compute_metrics(p):

    # Separate logits and labels
    pred, labels = p    
    
    # Return index of higher position (neg or positive residue) per row
    max_pred = np.argmax(pred, axis=-1)

    # Calculate probabilites to be used for AUC
    probs = torch.softmax(torch.tensor(pred), dim=-1)
    probs = probs[:, :, 1]

    # Create mask for unlabelled positions
    mask = labels != -100

    # Calculate metrics
    accuracy = accuracy_score(y_true=labels[mask], y_pred=max_pred[mask])
    recall = recall_score(y_true=labels[mask], y_pred=max_pred[mask])
    precision = precision_score(y_true=labels[mask], y_pred=max_pred[mask])
    f1 = f1_score(y_true=labels[mask], y_pred=max_pred[mask])
    mcc = matthews_corrcoef(y_true=labels[mask], y_pred=max_pred[mask])
    auc = roc_auc_score(y_true=labels[mask], y_score=probs[mask])

    return {'accuracy': accuracy, 'precision': precision, 'recall': recall, 'f1': f1, 'mcc': mcc, 'auc': auc}

model = AutoModelForTokenClassification.from_pretrained(checkpoint)

training_args = TrainingArguments(
    output_dir='/Users/harry/Documents/Data Science MSc/PROJECT/MScProject',
    # learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_strategy='epoch',
    load_best_model_at_end=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=preprocessed_train,
    eval_dataset=preprocessed_val,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] EsmForTokenClassification LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
classifier.bias           | MISSING    | 
classifier.weight         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/opt/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Mcc,Auc
1,0.634971,0.507439,0.854483,0.875000,0.013121,0.025854,0.096617,0.598338
2,0.575679,0.496760,0.856828,0.967742,0.028116,0.054645,0.151780,0.667750
3,0.543488,0.481742,0.863586,0.953488,0.076851,0.142238,0.249363,0.718026
4,0.562131,0.475584,0.867448,0.907692,0.110590,0.197160,0.290067,0.769014
5,0.519209,0.469356,0.868552,0.856250,0.128397,0.223309,0.300668,0.784789


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/opt/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


...

/opt/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/opt/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=130, training_loss=0.5670957418588491, metrics={'train_runtime': 1651.5081, 'train_samples_per_second': 1.253, 'train_steps_per_second': 0.079, 'total_flos': 94103146967040.0, 'train_loss': 0.5670957418588491, 'epoch': 5.0})

### Load pretrained classifier head

In [36]:
# Create simple 2 layer neural network in PyTorch to return logits per residue representing probability of each class
class PerResidueClassifier(nn.Module):
    def __init__(self, in_features=320, hidden_size=128, out_features=2):
        super().__init__()
        self.linear1 = nn.Linear(in_features, hidden_size)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout()
        self.linear2 = nn.Linear(hidden_size, out_features)
        
    def forward(self, embeddings):
        x = self.linear1(embeddings)
        x = self.relu(x)
        x = self.dropout(x)
        logits = self.linear2(x)
        
        return logits

checkpoint_dir = Path("/Users/harry/Documents/Data Science MSc/PROJECT/MScProject/clf_checkpoint")

clf_head = PerResidueClassifier()
clf_head.load_state_dict(torch.load(checkpoint_dir / 'best_clf.pt'))


<All keys matched successfully>

In [37]:
# Preprocess test set and generate embeddings using ESM2 as standard model
def preprocess_csv(csv_file):
    preprocessed_df = pd.read_csv(csv_file)

    # Mask n/a values with -100 
    preprocessed_df['Class'] = preprocessed_df['Class'].fillna(-100).astype('int32')
    preprocessed_df['Class'] = preprocessed_df['Class'].replace(-1, 0)
    
    # Sort values before aggregation
    preprocessed_df = preprocessed_df.sort_values(['Info_protein_id', 'Info_pos'])
    
    # Aggregate columns for wide format
    preprocessed_df = preprocessed_df.groupby(['Info_protein_id','Info_group'], as_index=False).agg(
        sequence=('Info_AA', ''.join), 
        label=('Class', list), 
        position=('Info_pos', list))

    # Apply sliding window
    preprocessed_df = sliding_window(preprocessed_df)

    return preprocessed_df

In [38]:
test_df = preprocess_csv('/Users/harry/Documents/Data Science MSc/PROJECT/MScProject/Target_1769.csv')

In [40]:
preprocessed_test = test_df.apply(preprocess, axis=1)

0     {'input_ids': [20, 8, 22, 15, 8, 7, 6, 10, 23,...
1     {'input_ids': [20, 5, 4, 22, 10, 8, 4, 20, 15,...
2     {'input_ids': [20, 14, 6, 10, 13, 6, 9, 11, 16...
3     {'input_ids': [20, 5, 10, 5, 7, 6, 12, 13, 4, ...
4     {'input_ids': [20, 5, 10, 11, 4, 10, 23, 23, 5...
5     {'input_ids': [20, 4, 6, 14, 8, 12, 6, 5, 19, ...
6     {'input_ids': [9, 18, 6, 4, 13, 6, 7, 11, 19, ...
7     {'input_ids': [20, 12, 13, 7, 8, 6, 15, 12, 10...
8     {'input_ids': [20, 11, 13, 16, 14, 14, 14, 8, ...
9     {'input_ids': [20, 4, 9, 11, 7, 10, 8, 7, 20, ...
10    {'input_ids': [20, 5, 9, 20, 12, 11, 9, 5, 5, ...
11    {'input_ids': [20, 4, 19, 8, 5, 13, 14, 8, 11,...
12    {'input_ids': [20, 5, 15, 11, 12, 5, 19, 13, 9...
13    {'input_ids': [20, 4, 20, 10, 11, 13, 14, 18, ...
14    {'input_ids': [20, 5, 15, 7, 15, 12, 15, 14, 4...
15    {'input_ids': [20, 12, 13, 7, 8, 6, 15, 12, 10...
16    {'input_ids': [20, 8, 9, 5, 15, 10, 4, 13, 14,...
17    {'input_ids': [20, 11, 14, 10, 15, 5, 5, 1